In [1]:
!pip install timm

import torch
import torch.nn as nn
import torch.optim as optim
import timm
import numpy as np
import os
import cv2
import random

from PIL import Image
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.utils.class_weight import compute_class_weight

In [2]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

In [3]:
DATASET_PATH = "/kaggle/input/datasets/sekhsujonhaque/cape-gooseberry-classification-dataset/classification_dataset"

In [4]:
class LabTransform:
    def __call__(self, img):
        img = np.array(img)
        img = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
        return Image.fromarray(img)

In [5]:
train_transforms = transforms.Compose([
    LabTransform(),
    transforms.Resize((300, 300)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(0.2,0.2,0.2,0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5], [0.5,0.5,0.5])
])

val_transforms = transforms.Compose([
    LabTransform(),
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5], [0.5,0.5,0.5])
])

In [6]:
train_dataset = datasets.ImageFolder(DATASET_PATH + "/train", transform=train_transforms)
val_dataset   = datasets.ImageFolder(DATASET_PATH + "/valid", transform=val_transforms)
test_dataset  = datasets.ImageFolder(DATASET_PATH + "/test", transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32)
test_loader  = DataLoader(test_dataset, batch_size=32)

print("Classes:", train_dataset.classes)

Classes: ['INTERMEDIA', 'MADURO', 'NO_APTA', 'VERDE']


In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

Using: cuda


In [8]:
labels = [label for _, label in train_dataset]

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)

class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

In [9]:
model = timm.create_model('efficientnet_b3', pretrained=True)
for param in model.parameters():
    param.requires_grad = False
for param in model.blocks[-5:].parameters():
    param.requires_grad = True
model.classifier = nn.Sequential(
    nn.BatchNorm1d(model.classifier.in_features),
    nn.Dropout(0.5),
    nn.Linear(model.classifier.in_features, 4)
)
model = model.to(device)

In [10]:
optimizer = optim.AdamW(model.parameters(), lr=0.00015)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

In [11]:
def evaluate(loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return 100 * correct / total

In [12]:
EPOCHS = 35
best_val = 0
patience = 5
counter = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    scheduler.step()

    val_acc = evaluate(val_loader)

    if val_acc > best_val:
        best_val = val_acc
        counter = 0
        torch.save(model.state_dict(), "/kaggle/working/best_model_lab_fixed.pth")
        print("Best model saved")
    else:
        counter += 1

    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss:.4f}, Val Acc: {val_acc:.2f}%")

    if counter >= patience:
        print("Early stopping triggered")
        break

Best model saved
Epoch 1/35, Loss: 62.6089, Val Acc: 45.61%
Best model saved
Epoch 2/35, Loss: 46.8083, Val Acc: 60.82%
Best model saved
Epoch 3/35, Loss: 38.2973, Val Acc: 62.57%
Best model saved
Epoch 4/35, Loss: 31.2603, Val Acc: 69.01%
Best model saved
Epoch 5/35, Loss: 25.7537, Val Acc: 70.18%
Epoch 6/35, Loss: 20.8861, Val Acc: 67.25%
Epoch 7/35, Loss: 19.5784, Val Acc: 66.67%
Best model saved
Epoch 8/35, Loss: 17.7204, Val Acc: 70.76%
Epoch 9/35, Loss: 15.3524, Val Acc: 67.25%
Best model saved
Epoch 10/35, Loss: 14.6500, Val Acc: 73.10%
Epoch 11/35, Loss: 14.1124, Val Acc: 70.18%
Epoch 12/35, Loss: 12.2189, Val Acc: 69.01%
Epoch 13/35, Loss: 11.0148, Val Acc: 67.25%
Epoch 14/35, Loss: 10.6917, Val Acc: 67.84%
Epoch 15/35, Loss: 9.7440, Val Acc: 66.08%
Early stopping triggered


In [13]:
model.load_state_dict(torch.load("/kaggle/working/best_model_lab_fixed.pth"))
model.eval()

test_acc = evaluate(test_loader)
print("FINAL LAB TEST ACCURACY:", test_acc)

FINAL LAB TEST ACCURACY: 73.46938775510205
